In [2]:
import pandas as pd
import json
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import re
import os

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Common function to check data for anomalies
def check_anomalies(df, name="Data"):
    print(f"\n{'=' * 60}")
    print(f"ANOMALY CHECK: {name}")
    print(f"{'=' * 60}")
    # Missing values
    print("\n1. Missing Values:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])
    # Duplicate rows
    print("\n2. Duplicate Records:")
    duplicates = df.duplicated().sum()
    print("Number of duplicate records:", duplicates)
    # Empty strings
    print("\n3. Empty String Values:")
    empty_values = (df.astype(str).apply(lambda x: x.str.strip() == "").sum())
    if empty_values.sum() == 0:
        print("No empty string values found.")
    else:
        print(empty_values[empty_values > 0])
    # Numeric columns - negative values
    print("\n4. Negative Numeric Values:")
    numeric_columns = df.select_dtypes(include="number").columns
    for column in numeric_columns:
        negative = (df[column] < 0).sum()
        if negative > 0:
            print(f"{column}: {negative} negative values")
    # Display summary
    print("\n5. Data Summary:")
    print(df.describe(include="all"))





In [4]:
# Common function to check data for anomalies
def check_anomalies(df, name="Data"):
    print(f"\n{'=' * 60}")
    print(f"ANOMALY CHECK: {name}")
    print(f"{'=' * 60}")
    # Missing values
    print("\n1. Missing Values:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])
    # Duplicate rows
    print("\n2. Duplicate Records:")
    duplicates = df.duplicated().sum()
    print("Number of duplicate records:", duplicates)
    # Empty strings
    print("\n3. Empty String Values:")
    empty_values = (df.astype(str).apply(lambda x: x.str.strip() == "").sum())
    if empty_values.sum() == 0:
        print("No empty string values found.")
    else:
        print(empty_values[empty_values > 0])
    # Numeric columns - negative values
    print("\n4. Negative Numeric Values:")
    numeric_columns = df.select_dtypes(include="number").columns
    for column in numeric_columns:
        negative = (df[column] < 0).sum()
        if negative > 0:
            print(f"{column}: {negative} negative values")
    # Display summary
    print("\n5. Data Summary:")
    print(df.describe(include="all"))





In [5]:
# 2. Parse CSV file
# ---------------------------------------------------------
def parse_csv_file(filename):
    print("\n\n===== CSV FILE =====")
    try:
        df = pd.read_csv(filename)
        print("\nExtracted Data:")
        print(df)
        check_anomalies(df, "CSV File")
        return df
    except Exception as e:
        print("Error reading CSV:", e)
        return None


In [6]:
# 3. Parse HTML file
# ---------------------------------------------------------
def parse_html_file(filename):
    print("\n\n===== HTML FILE =====")
    with open(filename, "r", encoding="utf-8") as file:
        html = file.read()
    soup = BeautifulSoup(html, "html.parser")
    records = []
    # Extract HTML tables
    tables = soup.find_all("table")
    for table in tables:
        rows = table.find_all("tr")
        headers = []
        for row_index, row in enumerate(rows):
            cells = row.find_all(["th", "td"])
            values = [cell.get_text(strip=True) for cell in cells]
            if row_index == 0:
                headers = values
            else:
                if values:
                    if headers and len(values) == len(headers):
                        records.append(dict(zip(headers, values)))
    if records:
        df = pd.DataFrame(records)

        print("\nExtracted Data:")
        print(df)
        check_anomalies(df, "HTML File")
        return df
    print("No table data found in HTML file.")
    return None
